In [ ]:
import torch
import pandas as pd
from transformers import GPT2LMHeadModel, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 데이터 불러오기

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ChatbotData.csv", encoding="utf-8")
df

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0
...,...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!,2
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.,2
11820,흑기사 해주는 짝남.,설렜겠어요.,2
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.,2


In [ ]:
df['Q'].str.len().max()

56

In [ ]:
df['A'].str.len().max()

76

input_ids : 입력 데이터

attention_mask : 패딩 지정. 텍스트 있는곳 1, pad = 0

labels : loss 계산시의 정답.패딩 부분이랑 질문은 -100 으로 두고 답변만 실제 토큰 id 남겨두고 label 이라고 지정

# 커스텀 Dataset 구성

In [ ]:
class ChatbotDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df) #텐서 계산 위해 길이 뽑아

    def __getitem__(self, idx):
        question = self.df.iloc[idx]["Q"]
        answer = self.df.iloc[idx]["A"]

        bos = self.tokenizer.bos_token_id
        eos = self.tokenizer.eos_token_id
        sep = self.tokenizer.convert_tokens_to_ids('<sep>') #sep 가 없다고 나와서 직접 지정

        q_tokens = self.tokenizer.encode(question, add_special_tokens=False)
        a_tokens = self.tokenizer.encode(answer, add_special_tokens=False)

        # 구조: <BOS> 질문 <SEP> 답변 <EOS>
        # 모델 입력 = 전체
        # loss 계산 = 답변 + EOS 부분만
        prompt_part = [bos] + q_tokens + [sep]
        answer_part = a_tokens + [eos]
        input_ids = prompt_part + answer_part

        input_ids = input_ids[:self.max_length]

        # labels: 질문 부분은 -100 (무시), 답변 부분만 학습
        labels = [-100] * len(prompt_part) + answer_part #crossentropyloss 연산 방식에 의해 answer 앞부분은 -100
        labels = labels[:self.max_length] #max_len 길이 만큼 truncate

        # 패딩
        pad_id = self.tokenizer.pad_token_id # 숫자 ID를 가져와야 텐서 변환 시 에러가 나지 않음 (id 가져와야 하니까)
        pad_len = self.max_length - len(input_ids) # 실제 시퀀스 길이 뒤에 패딩 부여
        attention_mask = [1] * len(input_ids) + [0] * pad_len #패딩 부분은 학습하면 안되니까
        input_ids = input_ids + [pad_id] * pad_len
        labels = labels + [-100] * pad_len

        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(labels),
        }


In [ ]:
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel

# skt/kogpt2-base-v2는 PreTrainedTokenizerFast를 사용해야 안깨진다고 함.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    bos_token='</s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>'
)

# 질문과 답변을 구분할 <sep> 토큰 추가
tokenizer.add_special_tokens({'additional_special_tokens': ['<sep>']})

model = GPT2LMHeadModel.from_pretrained('skt/kogpt2-base-v2')
model.resize_token_embeddings(len(tokenizer)) # 추가된 토큰만큼 임베딩 사이즈 조절

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(51201, 768)

## Train, Test, Val 분할

In [ ]:
# train / val / test 분할  8:1:1
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)


train_dataset = ChatbotDataset(train_df.reset_index(drop=True), tokenizer)
val_dataset = ChatbotDataset(val_df.reset_index(drop=True), tokenizer)
test_dataset = ChatbotDataset(test_df.reset_index(drop=True), tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

## 학습 준비

In [ ]:
from torch.optim import AdamW

EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=5e-5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Device: {device}")

Device: cuda


In [ ]:
for epoch in range(EPOCHS):
    # 학습
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # 검증
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1/3 | Train Loss: 2.8192 | Val Loss: 2.4280
Epoch 2/3 | Train Loss: 1.9068 | Val Loss: 2.2683
Epoch 3/3 | Train Loss: 1.3060 | Val Loss: 2.1713


## Test

In [ ]:
model.eval()
total_test_loss = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_test_loss += outputs.loss.item()

avg_test_loss = total_test_loss / len(test_loader)
print(f"Test Loss: {avg_test_loss:.4f}")

Test Loss: 2.1450


## Q-A 로 테스트

In [ ]:
model.eval()

question = "내 기분은 엉망이야"

input_text = tokenizer.bos_token + question + "<sep>"
input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device) #인풋 구성

# 생성
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_new_tokens=100,#최대 100 토큰까지만 생성 (답변 길이 조절)
        pad_token_id=tokenizer.pad_token_type_id, #지정한 pad 토큰 사용
        eos_token_id=tokenizer.eos_token_id,#eos 토큰 사용
        do_sample=True,#확률적 샘플링, 같은 질문도 미묘하게 다른 답 나오게 지정
        temperature=0.7, #높을수록 단어 분포가 평탄, 창의적 답변이 나옴
        top_k=50, #확률 상위 50개 토큰만 남기고 나머지는 후보에서 제외.지정 안해주면 엉뚱한 답 생성 가능
        top_p=0.9, #topk 는 확률 높은 단어 개수라면 topp 는 높은 확률부터 누적해서 90퍼 까지만 남김
        #top_p 의 경우 모델 학습 데이터에 따라, 확률 샘플링에 따라 유동적이기에 더 다양한 단어가 나올 수 있음.
    )

# 디코딩
response = tokenizer.decode(output[0], skip_special_tokens=False)
print(response)

</s> 내 기분은 엉망이야<sep> 저도 그래요.</s>
